# Diferencijalni manometar — predvidi, izračunaj, provjeri

**Poglavlje U03: hidrostatičko polje i manometrija**

Za priključke na istoj visini računamo razliku tlakova i istu vrijednost
rekonstruiramo hodom po stupcima fluida. Potom procjenjujemo osjetljivost
instrumenta i determinističku mjernu nesigurnost.


## 1. Predvidi

1. Ako se očitana razlika visina udvostruči, kako se mijenja $\Delta p$?
2. Što se događa s osjetljivošću kada su gustoće dvaju fluida gotovo jednake?
3. Hoće li zamjena predznaka očitanja zamijeniti predznak razlike tlakova?

Dogovorit ćemo $\Delta p=p_A-p_B$ i $\Delta h>0$ kada je desna razdjelnica viša.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
G = 9.81

def delta_p(rho_radni, rho_man, dh_m):
    return (rho_man - rho_radni) * G * dh_m

rho1, rho2, dh = 998.0, 13600.0, 42e-3
dp = delta_p(rho1, rho2, dh)
print(f"Δp = p_A - p_B = {dp/1000:.3f} kPa")


## 2. Izračunaj — neovisni hod po manometru

Uvedimo proizvoljnu dubinu lijeve razdjelnice $a$. Od priključka A idemo
dolje kroz radni fluid, zatim gore kroz manometarski fluid za $\Delta h$ i
konačno gore do priključka B. Proizvoljna dubina $a$ mora se poništiti.


In [ ]:
def tlak_hodom(p_A, rho_radni, rho_man, dh_m, a_m):
    if not (0 < dh_m < a_m):
        raise ValueError("Za ovu skicu mora vrijediti 0 < Δh < a.")
    p_lijeva_razdjelnica = p_A + rho_radni * G * a_m
    p_desna_razdjelnica = p_lijeva_razdjelnica - rho_man * G * dh_m
    p_B = p_desna_razdjelnica - rho_radni * G * (a_m - dh_m)
    return p_B

p_A = 120_000.0
dubine = np.linspace(0.08, 0.50, 8)
dp_hod = np.array([p_A - tlak_hodom(p_A, rho1, rho2, dh, a)
                   for a in dubine])
print("Rekonstrukcije Δp za različit proizvoljni a [Pa]:")
print(np.round(dp_hod, 6))

dh_mreza = np.linspace(-80e-3, 80e-3, 161)
fig, ax = plt.subplots(figsize=(6.8, 3.8))
for rho_m in (1100.0, 1800.0, 13600.0):
    ax.plot(1000*dh_mreza, delta_p(rho1, rho_m, dh_mreza),
            label=fr"$\rho_m={rho_m:.0f}$ kg/m³")
ax.set(xlabel=r"očitanje $\Delta h$ (mm)", ylabel=r"$p_A-p_B$ (Pa)",
       title="Osjetljivost ovisi o razlici gustoća")
ax.grid(ls=":", alpha=0.6)
ax.legend()
plt.show()


## 3. Provjeri — bilanca i mjerna nesigurnost

Za neovisne standardne nesigurnosti gustoća i očitanja koristimo parcijalne
derivacije izraza $\Delta p=(\rho_2-\rho_1)g\Delta h$.


In [ ]:
u_rho1, u_rho2, u_dh = 1.0, 20.0, 0.20e-3
u_dp = np.sqrt(
    (-G*dh*u_rho1)**2
    + (G*dh*u_rho2)**2
    + (G*(rho2-rho1)*u_dh)**2
)
print(f"Δp = {dp/1000:.3f} ± {u_dp/1000:.3f} kPa")

# Hod po fluidima, promjena orijentacije i jednake gustoće su neovisne provjere.
assert np.allclose(dp_hod, dp, rtol=1e-12, atol=1e-8)
assert np.isclose(delta_p(rho1, rho2, -dh), -dp, rtol=1e-12)
assert delta_p(rho1, rho1, dh) == 0.0
print("PASS: hod po stupcima, predznak očitanja i jednake gustoće su potvrđeni.")


## Granica modela

Izraz vrijedi za priključke na istoj visini, mirujuće homogene fluide i
jasno određene razdjelnice. Za različite visine priključaka treba provesti
puni hod po svim stupcima; za kapilare treba uključiti međupovršinske učinke.
